In [3]:
from pathlib import Path

import geopandas as gpd
import pandas as pd
import numpy as np
import rasterio
import matplotlib.pyplot as plt

from shapely.geometry import Point

# folders
DATA_RAW = Path("../data/raw/Thailand")
DATA_PROCESSED = Path("../data/processed")

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

THAILAND_CRS = "EPSG:32647"

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [4]:
def sample_raster_at_points(raster_path, points_gdf, band=1):
    """
    Read raster pixel values at each point location.
    Automatically converts the points to the raster CRS.
    """

    with rasterio.open(raster_path) as src:

        if points_gdf.crs != src.crs:
            pts = points_gdf.to_crs(src.crs)
        else:
            pts = points_gdf

        coords = [
            (geom.x, geom.y)
            for geom in pts.geometry
        ]

        values = np.array([
            v[band - 1]
            for v in src.sample(coords)
        ])

    return values

In [7]:
csv_path = DATA_RAW / "data_05010102-.csv"

df = pd.read_csv(csv_path)

print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

df.head()

Shape: (56177, 9)

Columns:
['invent_id', 'utm_e', 'utm_n', 'tambon', 'district', 'province', 'year_data', 'remark', 'remark_2']


,invent_id,utm_e,utm_n,tambon,district,province,year_data,remark,remark_2
0,1,894434,1385348,ด่านชุมพล,บ่อไร่,ตราด,0,NaN,NaN
1,2,897280,1384770,ด่านชุมพล,บ่อไร่,ตราด,0,NaN,NaN
2,3,898248,1384119,ด่านชุมพล,บ่อไร่,ตราด,0,NaN,NaN
3,4,904162,1384286,ด่านชุมพล,บ่อไร่,ตราด,0,NaN,NaN
4,5,908943,1373246,ท่ากุ่ม,เมืองตราด,ตราด,0,NaN,NaN


In [8]:
print(
    df[
        [
            "invent_id",
            "utm_e",
            "utm_n",
            "province",
            "district"
        ]
    ].head(10)
)

print("\nCoordinate summary:")

print(
    df[
        ["utm_e", "utm_n"]
    ].describe()
)

   invent_id   utm_e    utm_n province   district
0          1  894434  1385348     ตราด     บ่อไร่
1          2  897280  1384770     ตราด     บ่อไร่
2          3  898248  1384119     ตราด     บ่อไร่
3          4  904162  1384286     ตราด     บ่อไร่
4          5  908943  1373246     ตราด  เมืองตราด
5          6  524271  1479533  ราชบุรี    สวนผึ้ง
6          7  523829  1479678  ราชบุรี    สวนผึ้ง
7          8  523735  1479612  ราชบุรี    สวนผึ้ง
8          9  523920  1479955  ราชบุรี    สวนผึ้ง
9         10  523978  1479917  ราชบุรี    สวนผึ้ง

Coordinate summary:
              utm_e         utm_n
count  5.617700e+04  5.617700e+04
mean   6.043355e+05  1.572060e+06
std    1.238023e+05  5.186095e+05
min    3.528280e+05  6.249470e+05
25%    5.225680e+05  9.808270e+05
50%    6.082710e+05  1.896386e+06
75%    6.720000e+05  1.981990e+06
max    1.210411e+06  2.256843e+06


In [9]:
# Clean the coordinates
df["utm_e"] = pd.to_numeric(
    df["utm_e"],
    errors="coerce"
)

df["utm_n"] = pd.to_numeric(
    df["utm_n"],
    errors="coerce"
)

print("Original:", len(df))

df = df.dropna(
    subset=["utm_e", "utm_n"]
).copy()

df = df[
    (df["utm_e"] > 0) &
    (df["utm_n"] > 0)
].copy()

print("After invalid coordinate removal:", len(df))

Original: 56177
After invalid coordinate removal: 56177


In [10]:
# Remove duplicate coordinates
duplicates = df.duplicated(
    subset=["utm_e", "utm_n"]
).sum()

print("Duplicate coordinates:", duplicates)

df = df.drop_duplicates(
    subset=["utm_e", "utm_n"]
).reset_index(drop=True)

print("Unique landslide locations:", len(df))

Duplicate coordinates: 1356
Unique landslide locations: 54821


In [11]:
#Convert CSV into a GeoDataFrame
landslides = gpd.GeoDataFrame(
    df.copy(),

    geometry=gpd.points_from_xy(
        df["utm_e"],
        df["utm_n"]
    ),

    crs=THAILAND_CRS
)

landslides["label"] = 1

print("Landslide points:", len(landslides))
print("CRS:", landslides.crs)

landslides.head()

Landslide points: 54821
CRS: EPSG:32647


,invent_id,utm_e,utm_n,tambon,district,province,year_data,remark,remark_2,geometry,label
0,1,894434,1385348,ด่านชุมพล,บ่อไร่,ตราด,0,NaN,NaN,POINT (894434 1385348),1
1,2,897280,1384770,ด่านชุมพล,บ่อไร่,ตราด,0,NaN,NaN,POINT (897280 1384770),1
2,3,898248,1384119,ด่านชุมพล,บ่อไร่,ตราด,0,NaN,NaN,POINT (898248 1384119),1
3,4,904162,1384286,ด่านชุมพล,บ่อไร่,ตราด,0,NaN,NaN,POINT (904162 1384286),1
4,5,908943,1373246,ท่ากุ่ม,เมืองตราด,ตราด,0,NaN,NaN,POINT (908943 1373246),1


In [12]:
# Convert to latitude/longitude for checking
landslides_wgs84 = landslides.to_crs(
    "EPSG:4326"
)

landslides["longitude"] = (
    landslides_wgs84.geometry.x.values
)

landslides["latitude"] = (
    landslides_wgs84.geometry.y.values
)

print(
    landslides[
        [
            "utm_e",
            "utm_n",
            "latitude",
            "longitude"
        ]
    ].head()
)

    utm_e    utm_n   latitude   longitude
0  894434  1385348  12.507209  102.628059
1  897280  1384770  12.501639  102.654127
2  898248  1384119  12.495643  102.662935
3  904162  1384286  12.496406  102.717271
4  908943  1373246  12.396176  102.759742


In [13]:
#Check coordinates are actually in Thailand
valid = (
    landslides["latitude"].between(
        5.5,
        20.5
    )
    &
    landslides["longitude"].between(
        97.0,
        106.0
    )
)

print(
    "Valid Thailand locations:",
    valid.sum()
)

print(
    "Outside Thailand:",
    (~valid).sum()
)

print(
    "Valid percentage:",
    round(valid.mean() * 100, 2),
    "%"
)

Valid Thailand locations: 54821
Outside Thailand: 0
Valid percentage: 100.0 %


In [14]:
# Keep only valid points
landslides = (
    landslides[valid]
    .copy()
    .reset_index(drop=True)
)

print(
    "Usable landslide points:",
    len(landslides)
)

Usable landslide points: 54821


In [15]:
#Optional spatial duplicate filtering
GRID_SIZE = 30

landslides["grid_x"] = (
    landslides.geometry.x //
    GRID_SIZE
).astype(int)

landslides["grid_y"] = (
    landslides.geometry.y //
    GRID_SIZE
).astype(int)

before = len(landslides)

landslides = (
    landslides
    .drop_duplicates(
        subset=["grid_x", "grid_y"]
    )
    .copy()
)

landslides = landslides.drop(
    columns=["grid_x", "grid_y"]
)

print("Before spatial filtering:", before)

print(
    "After spatial filtering:",
    len(landslides)
)


Before spatial filtering: 54821
After spatial filtering: 53073


In [16]:
# 
MAX_LANDSLIDES = 10000

if len(landslides) > MAX_LANDSLIDES:

    landslides = landslides.sample(
        n=MAX_LANDSLIDES,
        random_state=RANDOM_SEED
    ).copy()

landslides = landslides.reset_index(
    drop=True
)

print(
    "Selected landslide points:",
    len(landslides)
)

Selected landslide points: 10000


In [17]:
# Create control/non-landslide points
minx, miny, maxx, maxy = (
    landslides.total_bounds
)

print(
    minx,
    miny,
    maxx,
    maxy
)


355194.0 625248.0 1209813.0 2252352.0


In [18]:
from shapely.geometry import Point


def generate_control_points(
    landslide_gdf,
    n_points,
    exclusion_distance=300,
    seed=42
):

    rng = np.random.default_rng(seed)

    minx, miny, maxx, maxy = (
        landslide_gdf.total_bounds
    )

    # Buffer known landslides
    exclusion_area = (
        landslide_gdf
        .geometry
        .buffer(exclusion_distance)
        .union_all()
    )

    controls = []

    attempts = 0

    max_attempts = n_points * 100

    while (
        len(controls) < n_points
        and
        attempts < max_attempts
    ):

        x = rng.uniform(
            minx,
            maxx
        )

        y = rng.uniform(
            miny,
            maxy
        )

        p = Point(x, y)

        if not exclusion_area.contains(p):

            controls.append(p)

        attempts += 1

    return gpd.GeoDataFrame(
        {
            "geometry": controls,
            "label": 0
        },
        crs=landslide_gdf.crs
    )

In [19]:
# Generate equal number of controls
non_landslides = generate_control_points(
    landslides,
    n_points=len(landslides),
    exclusion_distance=300,
    seed=RANDOM_SEED
)

print(
    "Landslides:",
    len(landslides)
)

print(
    "Controls:",
    len(non_landslides)
)

Landslides: 10000
Controls: 10000


In [20]:
# Convert control coordinates to latitude/longitude
controls_wgs84 = (
    non_landslides
    .to_crs("EPSG:4326")
)

non_landslides["longitude"] = (
    controls_wgs84.geometry.x.values
)

non_landslides["latitude"] = (
    controls_wgs84.geometry.y.values
)


In [21]:
# Add source metadata
landslides["source"] = (
    "Thailand_DMR_Landslide_Inventory"
)

non_landslides["source"] = (
    "Thailand_DMR_Control"
)

landslides["country"] = "Thailand"

non_landslides["country"] = (
    "Thailand"
)

In [22]:
# Combine classes
combined = pd.concat(
    [
        landslides,
        non_landslides
    ],
    ignore_index=True
)

combined = gpd.GeoDataFrame(
    combined,
    geometry="geometry",
    crs=THAILAND_CRS
)

print(
    combined["label"]
    .value_counts()
)

print(
    "CRS:",
    combined.crs
)

combined.head()

label
1    10000
0    10000
Name: count, dtype: int64
CRS: EPSG:32647


,invent_id,utm_e,utm_n,tambon,district,province,year_data,remark,remark_2,geometry,label,longitude,latitude,source,country
0,23956.0,631218.0,1982162.0,น้ำหมัน,ท่าปลา,อุตรดิตถ์,0.0,NaN,NaN,POINT (631218 1982162),1,100.238900,17.923531,Thailand_DMR_Landslide_Inventory,Thailand
1,42723.0,577754.0,976196.0,กรุงชิง,นบพิตำ,นครศรีธรรมราช,0.0,NaN,NaN,POINT (577754 976196),1,99.707065,8.830586,Thailand_DMR_Landslide_Inventory,Thailand
2,31550.0,526589.0,2123195.0,แม่ปั๋ง,พร้าว,เชียงใหม่,0.0,NaN,NaN,POINT (526589 2123195),1,99.252933,19.201974,Thailand_DMR_Landslide_Inventory,Thailand
3,31591.0,466498.0,2115911.0,ป่าแป๋,แม่แตง,เชียงใหม่,0.0,NaN,NaN,POINT (466498 2115911),1,98.681432,19.136043,Thailand_DMR_Landslide_Inventory,Thailand
4,33698.0,529901.0,2164429.0,ศรีดงเย็น,ไชยปราการ,เชียงใหม่,0.0,NaN,NaN,POINT (529901 2164429),1,99.285087,19.574568,Thailand_DMR_Landslide_Inventory,Thailand


In [37]:
# Inspect available factor rasters
FACTOR_DIR = (
    DATA_RAW /
    "shp_05010102-/SHP_05010102 ร่องรอยแผ่นดินถล่ม"
)

factor_files = sorted(
    FACTOR_DIR.glob("*.tif")
)

factor_files = [
    f for f in factor_files
    if not f.name.endswith(".ovr")
]

print(
    f"{len(factor_files)} factor rasters found:"
)

for f in factor_files:

    print(
        " ",
        f.name
    )

0 factor rasters found:


In [38]:
FACTOR_DIR = (
    DATA_RAW /
    "shp_05010102-" /
    "SHP_05010102 ร่องรอยแผ่นดินถล่ม"
)

print("Folder:", FACTOR_DIR)
print("Exists:", FACTOR_DIR.exists())
print("Is directory:", FACTOR_DIR.is_dir())

print("\nAll files inside:")

if FACTOR_DIR.exists():
    for f in FACTOR_DIR.iterdir():
        print(f.name)

Folder: ..\data\raw\Thailand\shp_05010102-\SHP_05010102 ร่องรอยแผ่นดินถล่ม
Exists: True
Is directory: True

All files inside:
LANDSLIDE_INVENTORY_MAP.cpg
LANDSLIDE_INVENTORY_MAP.dbf
LANDSLIDE_INVENTORY_MAP.dbf~
LANDSLIDE_INVENTORY_MAP.prj
LANDSLIDE_INVENTORY_MAP.sbn
LANDSLIDE_INVENTORY_MAP.sbx
LANDSLIDE_INVENTORY_MAP.shp
LANDSLIDE_INVENTORY_MAP.shp.xml
LANDSLIDE_INVENTORY_MAP.shx


In [39]:
import geopandas as gpd

shp_files = list(FACTOR_DIR.glob("*.shp"))

print("Shapefiles found:", len(shp_files))

for f in shp_files:
    print(f.name)

if shp_files:
    thailand_ls = gpd.read_file(shp_files[0])

    print("\nRows:", len(thailand_ls))
    print("CRS:", thailand_ls.crs)
    print("Columns:", thailand_ls.columns.tolist())

    display(thailand_ls.head())

Shapefiles found: 1
LANDSLIDE_INVENTORY_MAP.shp

Rows: 56177
CRS: EPSG:32647
Columns: ['INVENT_ID', 'UTM_E', 'UTM_N', 'TAMBON', 'DISTRICT', 'PROVINCE', 'YEAR_DATA', 'REMARK', 'REMARK_2', 'geometry']


,INVENT_ID,UTM_E,UTM_N,TAMBON,DISTRICT,PROVINCE,YEAR_DATA,REMARK,REMARK_2,geometry
0,56162,893805,1397299,บ่อพลอย,บ่อไร่,ตราด,NaN,None,None,POINT (893805 1397299)
1,56163,894847,1396666,บ่อพลอย,บ่อไร่,ตราด,NaN,None,None,POINT (894847 1396666)
2,56164,893805,1397299,บ่อพลอย,บ่อไร่,ตราด,NaN,None,None,POINT (893805 1397299)
3,56165,894847,1396666,บ่อพลอย,บ่อไร่,ตราด,NaN,None,None,POINT (894847 1396666)
4,56169,670929,2163084,ชนแดน,สองแคว,น่าน,NaN,None,None,POINT (670929 2163084)


In [40]:
from pathlib import Path

import geopandas as gpd
import pandas as pd
import numpy as np
import rasterio

from shapely.geometry import Point

DATA_RAW = Path("../data/raw/thailand")
DATA_PROCESSED = Path("../data/processed")

FACTOR_DIR = DATA_RAW / "Thailand_Factors"
FACTOR_DIR.mkdir(parents=True, exist_ok=True)

THAILAND_CRS = "EPSG:32647"

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [41]:
LANDSLIDE_DIR = (
    DATA_RAW /
    "shp_05010102-" /
    "SHP_05010102 ร่องรอยแผ่นดินถล่ม"
)

shp_files = list(LANDSLIDE_DIR.glob("*.shp"))

print("Shapefiles found:", len(shp_files))

for f in shp_files:
    print(f.name)

landslides = gpd.read_file(shp_files[0])

print("Rows:", len(landslides))
print("CRS:", landslides.crs)
print("Columns:", landslides.columns.tolist())

display(landslides.head())

Shapefiles found: 1
LANDSLIDE_INVENTORY_MAP.shp
Rows: 56177
CRS: EPSG:32647
Columns: ['INVENT_ID', 'UTM_E', 'UTM_N', 'TAMBON', 'DISTRICT', 'PROVINCE', 'YEAR_DATA', 'REMARK', 'REMARK_2', 'geometry']


,INVENT_ID,UTM_E,UTM_N,TAMBON,DISTRICT,PROVINCE,YEAR_DATA,REMARK,REMARK_2,geometry
0,56162,893805,1397299,บ่อพลอย,บ่อไร่,ตราด,NaN,None,None,POINT (893805 1397299)
1,56163,894847,1396666,บ่อพลอย,บ่อไร่,ตราด,NaN,None,None,POINT (894847 1396666)
2,56164,893805,1397299,บ่อพลอย,บ่อไร่,ตราด,NaN,None,None,POINT (893805 1397299)
3,56165,894847,1396666,บ่อพลอย,บ่อไร่,ตราด,NaN,None,None,POINT (894847 1396666)
4,56169,670929,2163084,ชนแดน,สองแคว,น่าน,NaN,None,None,POINT (670929 2163084)


In [42]:
if landslides.crs is None:
    landslides = landslides.set_crs(THAILAND_CRS)

elif landslides.crs.to_epsg() != 32647:
    landslides = landslides.to_crs(THAILAND_CRS)

print(landslides.crs)

EPSG:32647


In [43]:
print(landslides.geom_type.value_counts())

Point    56177
Name: count, dtype: int64


In [44]:
if not all(landslides.geometry.geom_type == "Point"):
    landslides["geometry"] = landslides.geometry.centroid

In [45]:
landslides["label"] = 1

In [46]:
landslides["x"] = landslides.geometry.x.round(1)
landslides["y"] = landslides.geometry.y.round(1)

location_counts = (
    landslides
    .groupby(["x", "y"])
    .size()
    .reset_index(name="record_count")
)

landslides = landslides.merge(
    location_counts,
    on=["x", "y"],
    how="left"
)

print(
    landslides["record_count"]
    .value_counts()
    .sort_index()
    .head(20)
)

record_count
1    53521
2     2490
3      162
4        4
Name: count, dtype: int64


In [47]:
positive = (
    landslides
    .sort_values("record_count", ascending=False)
    .drop_duplicates(subset=["x", "y"])
    .copy()
)

print("Unique landslide locations:", len(positive))

Unique landslide locations: 54821


In [48]:
MAX_POSITIVES = 10000

if len(positive) > MAX_POSITIVES:
    positive = positive.sample(
        n=MAX_POSITIVES,
        random_state=RANDOM_SEED
    )

positive = positive.reset_index(drop=True)

print("Selected positives:", len(positive))

Selected positives: 10000


In [49]:
def generate_control_points(
    positive_gdf,
    n_points,
    exclusion_distance=300,
    seed=42
):
    rng = np.random.default_rng(seed)

    minx, miny, maxx, maxy = positive_gdf.total_bounds

    exclusion = (
        positive_gdf.geometry
        .buffer(exclusion_distance)
        .union_all()
    )

    controls = []

    max_attempts = n_points * 100
    attempts = 0

    while len(controls) < n_points and attempts < max_attempts:
        x = rng.uniform(minx, maxx)
        y = rng.uniform(miny, maxy)

        p = Point(x, y)

        if not exclusion.contains(p):
            controls.append(p)

        attempts += 1

    return gpd.GeoDataFrame(
        {
            "geometry": controls,
            "label": 0,
            "record_count": 0
        },
        crs=positive_gdf.crs
    )

In [50]:
controls = generate_control_points(
    positive,
    n_points=len(positive),
    exclusion_distance=300,
    seed=RANDOM_SEED
)

print("Positive:", len(positive))
print("Controls:", len(controls))

Positive: 10000
Controls: 10000


In [51]:
samples = pd.concat(
    [
        positive,
        controls
    ],
    ignore_index=True
)

samples = gpd.GeoDataFrame(
    samples,
    geometry="geometry",
    crs=THAILAND_CRS
)

print(samples["label"].value_counts())

label
1    10000
0    10000
Name: count, dtype: int64


In [52]:
samples_wgs84 = samples.to_crs("EPSG:4326")

samples["longitude"] = samples_wgs84.geometry.x.values
samples["latitude"] = samples_wgs84.geometry.y.values

In [53]:
def sample_raster_at_points(
    raster_path,
    points_gdf,
    band=1
):
    with rasterio.open(raster_path) as src:

        if points_gdf.crs != src.crs:
            pts = points_gdf.to_crs(src.crs)
        else:
            pts = points_gdf

        coords = [
            (geom.x, geom.y)
            for geom in pts.geometry
        ]

        values = np.array([
            v[band - 1]
            for v in src.sample(coords)
        ])

    return values

In [54]:
factor_files = sorted(
    FACTOR_DIR.glob("*.tif")
)

factor_files = [
    f for f in factor_files
    if not f.name.endswith(".ovr")
]

print(
    f"{len(factor_files)} factor rasters found:"
)

for f in factor_files:
    print(" ", f.name)

0 factor rasters found:
